# Assignment 1 & Assignment 2 Consolidated Notebook
***Elise Wallace***

California State University San Marcos - M.S. Business Analytics Program

---

## Goal of the Exercise

It is important to keep the overarching goal of this exercise, which is to calculate the safety stock by SKU, where the inventory type is a finished good (FG) and the stocking type is made-to-stock (MTS).

It is also important to notate the formula for calculating safety stock, which is below:

$$SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

Knowing this, we know that the key features of this dataset would include the SKU, inventory type, stocking type, demand/quantity, and lead time.

## Importing Packages and Data


In [21]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import norm
from scipy.stats.mstats import mquantiles
from plotnine import *

transaction_data = pd.read_csv('../data/transaction_data.csv')
meta_data = pd.read_csv('../data/meta_data.csv')


# Part 1: Data Cleaning, Feature Engineering & Exploratory Subsetting (Assignment 1)

## Question 1: DataFrame Formatting

As you load and inspect your transaction dataframe, you'll notice the column names don't follow a consistent convention. Make the necessary changes to ensure all column names follow the same convention as the metadata.

### Answer 1:

#### Headers for meta_data
We want to first get the headers for the meta data in order to identify what needs to be corrected or matched. We also want to get a description of the feature/header for our actual transactional dataset.

We want to display the metadata table first.


In [22]:
meta_data


,feature,description
0,sku_number,Eight character stock keeping unit number
1,inventory_type,Types of inventory including finished good - w...
2,stocking_type,Stockign type policy including make to stock -...
3,lead_time,Total lead time in days
4,unit_price,Unit price US dollar value
5,manufacturing_site,Manufacturing site ID starting with country co...
6,division_code,Four character business division ID
7,transaction_date,Date of Transaction in Year-Month-Day format
8,order_quantity,Order quantity - rounded up for convenience


#### Meta_data Analysis

After looking at the table, we see that there are 9 features with the following data types:

| Feature | Data Type |
| -------- | ---------- |
| sku_number | Qualitative Nominal |
| inventory_type | Qualitative Ordinal |
| stocking_type | Qualitative Nominal |
| lead_time | Quantitative Discrete |
| unit_price | Quantitative Continuous |
| manufacturing_site | Qualitative Nominal |
| division_code | Qualitative Nominal |
| transaction_date | Qualitative Ordinal |
| order_quantity | Quantitative Discrete |

Now, we'll check the transaction_data to see if the headers and the data types match.


In [23]:
transaction_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 711662 entries, 0 to 711661
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          711249 non-null  str    
 1   inventory_type      698248 non-null  str    
 2   stocking_type       697858 non-null  str    
 3   leadTime            711662 non-null  int64  
 4   unit_price          711662 non-null  float64
 5   manufacturing_site  573584 non-null  str    
 6   division Code       661808 non-null  str    
 7   transaction_date    711662 non-null  str    
 8   Order-Quantity      711662 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 48.9 MB


#### Cross-Check & Creating Dictionary

Now we see that generally, all of the data type matches. The only one which we may need to perform some transformation on is 'transaction_date'.

As far as the headers, it looks like 'lead_time', 'division_code', and 'order_quantity' need to be updated. But just in case, let's go ahead and rename the columns by overwriting the existing column names.


In [24]:
transaction_title = ['sku_number', 'inventory_type', 'stocking_type', 'lead_time', 'unit_price', 'manufacturing_site', 'division_code', 'transaction_date', 'order_quantity']
transaction_data.columns = transaction_title


#### Validation for new column names

Next, we will look at the transaction data info to ensure that the correct headers were used.


In [25]:
transaction_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 711662 entries, 0 to 711661
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_number          711249 non-null  str    
 1   inventory_type      698248 non-null  str    
 2   stocking_type       697858 non-null  str    
 3   lead_time           711662 non-null  int64  
 4   unit_price          711662 non-null  float64
 5   manufacturing_site  573584 non-null  str    
 6   division_code       661808 non-null  str    
 7   transaction_date    711662 non-null  str    
 8   order_quantity      711662 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 48.9 MB


## Question 2: Numeric Anomalies & Cleaning

Inspect the numeric features of your dataframe with the describe function. Provide a summary of these features, and in case there are any oddities/anomalies in your dataframe, locate and correct them. Please include a note on any oddities you find and a rationale for your suggested correction.


In [26]:
transaction_data.describe()


,lead_time,unit_price,order_quantity
count,711662.000000,711662.000000,711662.000000
mean,25.771684,797.880175,79.788738
std,10.704492,375.268727,41.964285
min,14.000000,-1015.534324,-121.000000
25%,28.000000,982.030704,63.000000
50%,28.000000,996.396930,93.000000
75%,28.000000,1004.287522,109.000000
max,2800.000000,1029.941378,198.000000


### Answer 2:

We see that after running the describe function, there are negative numbers in the 'unit_price' and 'order_quantity'.

We also see a maximum lead time of 2800. We would want to also investigate any lead times which are beyond 3 standard deviations away from the mean.

#### Order Quantity less than 0

We see that we have 36,437 observations that have a negative order quantity. Looking at the inventory type, we see a mix of finished goods (FG), work-in-progress (WIP), and raw materials (RM).

These are most likely data entry errors, so we would take the absolute value of the negative values.


In [27]:
transaction_data.query('order_quantity < 0').describe()


,lead_time,unit_price,order_quantity
count,36437.000000,36437.000000,36437.000000
mean,16.616763,88.066587,-7.679337
std,4.304398,63.908562,6.532411
min,14.000000,32.909604,-121.000000
25%,14.000000,48.138280,-11.000000
50%,14.000000,53.110669,-6.000000
75%,21.000000,102.912212,-3.000000
max,28.000000,1011.818547,-1.000000


In [28]:
transaction_data_clean = transaction_data.copy()
transaction_data_clean['order_quantity'] = transaction_data_clean['order_quantity'].abs()
transaction_data_clean.describe()


,lead_time,unit_price,order_quantity
count,711662.000000,711662.000000,711662.00000
mean,25.771684,797.880175,80.57510
std,10.704492,375.268727,40.43386
min,14.000000,-1015.534324,0.00000
25%,28.000000,982.030704,63.00000
50%,28.000000,996.396930,93.00000
75%,28.000000,1004.287522,109.00000
max,2800.000000,1029.941378,198.00000


#### Order Quantity is 0

We also want to make sure that our data does not contain any order quantities where it is 0. Including these can often skew both the order quantity mean, and the lead time. If no actual order quantity was placed, we would consider this as a data entry error, or it could also indicate a cancellation. Rather than removing the transaction, it could be well-intended for the company's policy to still keep the transaction record, but update the quantity to 0. Nonetheless, for the purposes of our goal, it should be **omitted**.


In [29]:
transaction_data_clean.query('order_quantity == 0').describe()


,lead_time,unit_price,order_quantity
count,4148.000000,4148.000000,4148.0
mean,16.738910,80.791172,0.0
std,4.753034,55.894166,0.0
min,14.000000,32.909604,0.0
25%,14.000000,48.131864,0.0
50%,14.000000,52.730241,0.0
75%,21.000000,97.850171,0.0
max,28.000000,225.605428,0.0


In [30]:
transaction_data_clean = transaction_data_clean.query('order_quantity != 0')
transaction_data_clean.describe()


,lead_time,unit_price,order_quantity
count,707514.000000,707514.000000,707514.000000
mean,25.824641,802.084311,81.047494
std,10.707211,372.292255,40.077371
min,14.000000,-1015.534324,1.000000
25%,28.000000,982.703171,64.000000
50%,28.000000,996.598691,93.000000
75%,28.000000,1004.506165,109.000000
max,2800.000000,1029.941378,198.000000


#### Unit Price less than 0

For the unit pricing, similar to order quantity, we would attribute this as data-entry issues and will take the absolute value of the unit price.


In [31]:
transaction_data_clean.query('unit_price < 0').describe()


,lead_time,unit_price,order_quantity
count,10.000000,10.000000,10.000000
mean,27.300000,-920.553034,98.600000
std,2.213594,253.867643,25.695655
min,21.000000,-1015.534324,34.000000
25%,28.000000,-1004.778647,92.000000
50%,28.000000,-999.283847,107.000000
75%,28.000000,-995.216109,113.500000
max,28.000000,-198.417515,123.000000


In [32]:
transaction_data_clean['unit_price'] = transaction_data_clean['unit_price'].abs()
transaction_data_clean.describe()


,lead_time,unit_price,order_quantity
count,707514.000000,707514.000000,707514.000000
mean,25.824641,802.110333,81.047494
std,10.707211,372.236187,40.077371
min,14.000000,32.909604,1.000000
25%,28.000000,982.703171,64.000000
50%,28.000000,996.598691,93.000000
75%,28.000000,1004.506165,109.000000
max,2800.000000,1029.941378,198.000000


#### Unit price = 0

Next, we also want to look at anything with a unit price of 0. There is **no data** with a unit price of 0. No further action is needed.


In [33]:
transaction_data_clean.query('unit_price == 0').describe()


,lead_time,unit_price,order_quantity
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
25%,NaN,NaN,NaN
50%,NaN,NaN,NaN
75%,NaN,NaN,NaN
max,NaN,NaN,NaN


#### Lead time greater than 3 standard deviations from the mean (outliers)

While the describe function from pandas already gave us the values for the mean and standard deviation for lead time, I don't want to manually write down and calculate the number to filter. Thus, I've also loaded numpy to perform the calculation, set 2 variables called 'lt_3std' and 'lt_mean' and using those variables to apply my filter.


In [34]:
lt_3std = 3*np.std(transaction_data_clean['lead_time'])
lt_mean = np.mean(transaction_data_clean['lead_time'])
transaction_data_clean.query('abs(lead_time - @lt_mean) > @lt_3std').describe()


,lead_time,unit_price,order_quantity
count,10.00000,10.000000,10.000000
mean,2520.00000,814.230902,84.900000
std,590.29183,402.870461,39.478405
min,1400.00000,47.259863,13.000000
25%,2800.00000,995.972464,72.750000
50%,2800.00000,1000.637763,96.000000
75%,2800.00000,1008.420766,114.750000
max,2800.00000,1017.014895,123.000000


##### Checking for any lead times with a '0' value


In [35]:
transaction_data_clean.query('lead_time == 0').describe()


,lead_time,unit_price,order_quantity
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
25%,NaN,NaN,NaN
50%,NaN,NaN,NaN
75%,NaN,NaN,NaN
max,NaN,NaN,NaN


##### Lead Time Corrections

Based on the lead times that were in the transactional data, it is **extremely** unlikely that any lead time would be 3.5 to 7.7 years. Looking at the pattern, it looks like the lead times for the outliers were multiplied by 100. We would divide these outliers by 100.


In [36]:
outlier_condition = (transaction_data_clean['lead_time'] - lt_mean).abs() > lt_3std
transaction_data_clean.loc[outlier_condition, 'lead_time'] = (transaction_data_clean.loc[outlier_condition, 'lead_time'] / 100)
transaction_data_clean.describe()


,lead_time,unit_price,order_quantity
count,707514.000000,707514.000000,707514.000000
mean,25.789379,802.110333,81.047494
std,4.720666,372.236187,40.077371
min,14.000000,32.909604,1.000000
25%,28.000000,982.703171,64.000000
50%,28.000000,996.598691,93.000000
75%,28.000000,1004.506165,109.000000
max,28.000000,1029.941378,198.000000


## Question 3 — Missing Value (NaN) Handling

Your transactional dataset includes many missing values. Find and address (in your dataframe) the missing values by column. Again, remember to note the NaN values you found and provide a rationale for your suggested action.

### Answer 3:
We want to go through every column and sum the total numbers of NaN values in each column. Since we also REMOVED some rows from the previous step(s), we would also re-run the isna().sum() function after removing to get the 'corrected' values for the counts.

#### Summary of NaN counts for each feature after filtering from Question 2

| Feature | Count of NaN |
| -------- | ---------- |
| sku_number | 405 |
| inventory_type | 13,326 |
| stocking_type | 13,743 |
| lead_time | 0 |
| unit_price | 0 |
| manufacturing_site | 137,280 |
| division_code | 49,568 |
| transaction_date | 0 |
| order_quantity | 0 |

#### Summary of Decisions for each feature

For all of the categorical data, we filled all of the NaN values with an 'Unknown' category.

Lead time, unit price, transaction date, and order quantity require no transformation, filling, or removing of NaN values, since there are **no** NaN values.


In [37]:
for col in ['sku_number', 'inventory_type', 'stocking_type', 'manufacturing_site', 'division_code']:
    transaction_data_clean[col] = transaction_data_clean[col].fillna('Unknown')

transaction_data_clean.isna().sum()


sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

## Question 4 — Extracting The Country Values

To apply broader breakdowns of your transactional dataset, it will be helpful to add a country column. Recall that the metadata indicates the country information is somewhat captured in the site column. Create a new country column by using the manufacturing_site information. To do so, you will need the helper function str.startswith(), which is defined for a pandas Series object. As the documentation states, str.startswith() tests if the start of each string element matches a given pattern.

### Answer 4:
From the metadata, we know that the countries that we are interested in are USA, Canada, Poland, and Japan.

#### Find unique values under manufacturing site
Next, we want to create a list of the manufacturing sites. We see that there is a pattern, where PL = Poland, US = USA, CA = Canada, and JP = Japan. Rather than mapping out each unique value, we use this pattern to see if we can use the 'start with' function. Ideally, we'd like to just see the values of the first 2 characters of the string to then map it to a dictionary.


In [38]:
transaction_data_clean['manufacturing_site'].unique()


<StringArray>
['Unknown',   'PL-38',   'PL-B2',   'CA-B4',   'CA-4C',   'US-6B',   'CA-14',
   'PL-8F',   'JP-84',   'PL-0A',   'PL-A2',   'JP-96',   'JP-38',   'PL-2D',
   'PL-F8',   'US-2B',   'US-3A',   'JP-F5',   'PL-96',   'JP-67']
Length: 20, dtype: str

#### Creating a dictionary & Mapping to new country

In order to add a column based on the values of the manufacturing site, we create a dictionary first.

Creates a new column named 'country', we take it from the manufacturing site, look at the string of the first 2 characters, and map it to our dictionary.


In [39]:
country_map = {'CA' : 'Canada', 'JP' : 'Japan', 'PL' : 'Poland', 'US' : 'USA'}
transaction_data_clean['country'] = transaction_data_clean['manufacturing_site'].str[:2].map(country_map)
transaction_data_clean.sample(10)


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
470001,BF76352C,WIP,MTS,28,102.595111,PL-8F,C04F,2024-12-23,9,Poland
637064,3EEB8E26,FG,MTS,28,996.987714,CA-B4,A786,2025-09-06,133,Canada
592869,90D5E95A,FG,MTS,28,1011.266627,CA-B4,D9AC,2025-06-30,75,Canada
555063,687B7E45,FG,MTO,21,212.614781,Unknown,C04F,2025-05-02,36,NaN
26914,70201289,FG,MTS,28,983.767977,CA-B4,64C7,2023-02-11,108,Canada
501088,85D13161,WIP,MTS,28,995.450936,CA-B4,D9AC,2025-02-09,107,Canada
414424,292E3348,FG,MTS,28,984.713211,PL-8F,F049,2024-09-29,91,Poland
296469,6CB28188,FG,MTS,28,1002.227524,US-6B,B395,2024-04-01,77,USA
628526,AAAD8A64,RM,Unknown,28,1003.750430,Unknown,C04F,2025-08-24,95,NaN
479681,8C144104,WIP,MTS,28,991.764628,PL-B2,F049,2025-01-07,121,Poland


## Question 5 — Helpful Dimensional Information

It's time to focus on the subset of this dataset we need. Since we ultimately want to calculate safety stock, focus on finished goods designated as 'make to stock.' Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).

+ How many unique SKUs are we working with?
+ How many unique Manufacturing Sites are we working with?
+ How many Divisions are we working with?
+ What are the top and bottom 10 transactions by order quantity?
+ What are the top and bottom 10 transactions by total sales value?

### Answer 5:
#### Creating a filtered subset
We want to create a new subset called 'MTS_data' where we are working with the transactions where the stocking_type is only 'MTS' and the inventory_type is 'FG'.


In [40]:
MTS_data = transaction_data_clean.query('stocking_type == "MTS" and inventory_type == "FG"').copy()
MTS_data


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
1,FED6CA5E,FG,MTS,28,1019.701380,PL-38,1536,2023-01-01,63,Poland
7,84FCC369,FG,MTS,28,977.181212,Unknown,D9AC,2023-01-01,90,NaN
8,7E0B9A17,FG,MTS,28,999.436684,US-6B,BE87,2023-01-01,99,USA
12,8A1F415E,FG,MTS,28,1005.366262,CA-B4,1C8A,2023-01-01,78,Canada
13,90D5E95A,FG,MTS,14,41.499619,PL-8F,1536,2023-01-01,7,Poland
...,...,...,...,...,...,...,...,...,...,...
711638,7E0B9A17,FG,MTS,28,991.088830,PL-B2,CEA3,2025-12-30,100,Poland
711643,00874431,FG,MTS,28,990.146733,PL-B2,D9AC,2025-12-30,125,Poland
711644,63C4377B,FG,MTS,14,37.043908,CA-4C,A786,2025-12-30,9,Canada
711654,C8EADE59,FG,MTS,28,1000.570170,Unknown,F049,2025-12-30,94,NaN


#### Unique SKUs Count
To see the unique number of SKUs in our filtered subset, we want to do a value_counts. This value_counts will also automatically *ignore* NaN entries. We also want to find the LENGTH, so we use the len function.

We have 375 unique SKUs.


In [41]:
len(MTS_data['sku_number'].value_counts())


376

#### Unique manufacturing sites
We perform the same exercise with the SKUs. There are 19 unique manufacturing sites.


In [42]:
len(MTS_data['manufacturing_site'].value_counts())


20

#### Unique divisions
There are 54 unique divisions for MTS transactions.


In [43]:
len(MTS_data['division_code'].value_counts())


55

#### Finding top and bottom 10 transactions by order quantity


In [44]:
# Top 10 transactions by order quantity
MTS_data.sort_values('order_quantity', ascending = False).head(10)


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
33499,7E0B9A17,FG,MTS,28,991.088830,CA-B4,CEA3,2023-02-21,198,Canada
185631,FDDDA064,FG,MTS,28,1010.068090,JP-38,C04F,2023-10-12,194,Japan
180871,285D1D8D,FG,MTS,28,1000.289315,CA-B4,D9AC,2023-10-05,186,Canada
406049,E045018B,FG,MTS,28,995.605558,PL-0A,D9AC,2024-09-16,184,Poland
393301,578DD826,FG,MTS,28,1019.132513,CA-B4,DC0A,2024-08-27,181,Canada
155365,B876EA18,FG,MTS,28,1002.119781,Unknown,B395,2023-08-27,180,NaN
278589,5C704F7E,FG,MTS,28,987.620742,PL-8F,1536,2024-03-04,179,Poland
641161,5E589032,FG,MTS,28,1006.262150,Unknown,CEA3,2025-09-13,178,NaN
642523,7D1D8544,FG,MTS,28,996.942519,JP-38,A786,2025-09-15,178,Japan
671624,ACC4CB10,FG,MTS,28,998.841700,PL-B2,CB7E,2025-10-30,178,Poland


In [45]:
# Bottom 10 transactions by order quantity (excluding zero/negative)
MTS_data.query('order_quantity > 0').sort_values('order_quantity').head(10)


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
466913,2F0EF6CA,FG,MTS,14,52.519041,Unknown,1C8A,2024-12-18,1,NaN
414704,9EC6C209,FG,MTS,14,55.435034,PL-B2,BE87,2024-09-29,1,Poland
236000,4734F298,FG,MTS,14,52.431935,JP-38,CEA3,2023-12-29,1,Japan
647868,5FAD7E6B,FG,MTS,14,49.513551,PL-8F,CB7E,2025-09-23,1,Poland
567797,5E6E557A,FG,MTS,14,52.950537,PL-B2,CB7E,2025-05-22,1,Poland
567780,F2A4F89E,FG,MTS,21,222.476530,US-3A,BE87,2025-05-22,1,USA
493536,00874431,FG,MTS,14,55.489123,JP-38,1C8A,2025-01-28,1,Japan
241945,D5B4B986,FG,MTS,14,57.241949,PL-8F,1590,2024-01-07,1,Poland
372006,AF837059,FG,MTS,21,193.928187,PL-B2,CB7E,2024-07-26,1,Poland
32891,8965E908,FG,MTS,14,45.025429,PL-B2,CB7E,2023-02-20,1,Poland


#### Adding total sales value column
We add a new column titled 'total_sales_value' and multiply the order quantity by the unit price.


In [46]:
MTS_data['total_sales_value'] = MTS_data['order_quantity']*MTS_data['unit_price']
MTS_data


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
1,FED6CA5E,FG,MTS,28,1019.701380,PL-38,1536,2023-01-01,63,Poland,64241.186944
7,84FCC369,FG,MTS,28,977.181212,Unknown,D9AC,2023-01-01,90,NaN,87946.309077
8,7E0B9A17,FG,MTS,28,999.436684,US-6B,BE87,2023-01-01,99,USA,98944.231693
12,8A1F415E,FG,MTS,28,1005.366262,CA-B4,1C8A,2023-01-01,78,Canada,78418.568415
13,90D5E95A,FG,MTS,14,41.499619,PL-8F,1536,2023-01-01,7,Poland,290.497336
...,...,...,...,...,...,...,...,...,...,...,...
711638,7E0B9A17,FG,MTS,28,991.088830,PL-B2,CEA3,2025-12-30,100,Poland,99108.882956
711643,00874431,FG,MTS,28,990.146733,PL-B2,D9AC,2025-12-30,125,Poland,123768.341576
711644,63C4377B,FG,MTS,14,37.043908,CA-4C,A786,2025-12-30,9,Canada,333.395170
711654,C8EADE59,FG,MTS,28,1000.570170,Unknown,F049,2025-12-30,94,NaN,94053.595963


#### Top and Bottom 10 transactions by total sales value


In [47]:
# Top 10 by total sales value
MTS_data.sort_values('total_sales_value', ascending = False).head(10)


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
33499,7E0B9A17,FG,MTS,28,991.088830,CA-B4,CEA3,2023-02-21,198,Canada,196235.588252
185631,FDDDA064,FG,MTS,28,1010.068090,JP-38,C04F,2023-10-12,194,Japan,195953.209533
180871,285D1D8D,FG,MTS,28,1000.289315,CA-B4,D9AC,2023-10-05,186,Canada,186053.812682
393301,578DD826,FG,MTS,28,1019.132513,CA-B4,DC0A,2024-08-27,181,Canada,184462.984874
406049,E045018B,FG,MTS,28,995.605558,PL-0A,D9AC,2024-09-16,184,Poland,183191.422709
155365,B876EA18,FG,MTS,28,1002.119781,Unknown,B395,2023-08-27,180,NaN,180381.560506
669165,8ED02F71,FG,MTS,28,1013.010673,JP-67,Unknown,2025-10-26,177,Japan,179302.889117
641161,5E589032,FG,MTS,28,1006.262150,Unknown,CEA3,2025-09-13,178,NaN,179114.662782
253147,6B527587,FG,MTS,28,1011.589319,PL-38,BE87,2024-01-24,176,Poland,178039.720095
671624,ACC4CB10,FG,MTS,28,998.841700,PL-B2,CB7E,2025-10-30,178,Poland,177793.822599


In [48]:
# Bottom 10 by total sales value
MTS_data.sort_values('total_sales_value').head(10)


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,total_sales_value
261559,B76BF308,FG,MTS,14,34.391014,Unknown,D9AC,2024-02-06,1,NaN,34.391014
335738,B76BF308,FG,MTS,14,34.391014,PL-8F,1C8A,2024-05-31,1,Poland,34.391014
621579,B76BF308,FG,MTS,14,34.391014,PL-38,CB7E,2025-08-13,1,Poland,34.391014
53888,B76BF308,FG,MTS,14,34.391014,Unknown,D9AC,2023-03-24,1,NaN,34.391014
183845,B76BF308,FG,MTS,14,34.391014,PL-B2,929A,2023-10-09,1,Poland,34.391014
403379,63C4377B,FG,MTS,14,37.043908,CA-B4,D9AC,2024-09-12,1,Canada,37.043908
351906,63C4377B,FG,MTS,14,37.043908,PL-8F,D9AC,2024-06-25,1,Poland,37.043908
551023,63C4377B,FG,MTS,14,37.043908,PL-B2,1536,2025-04-26,1,Poland,37.043908
285368,63C4377B,FG,MTS,14,37.043908,PL-8F,C04F,2024-03-15,1,Poland,37.043908
370981,63C4377B,FG,MTS,14,37.043908,PL-B2,64C7,2024-07-24,1,Poland,37.043908


# Part 2: SKU Aggregation, Parametric & Non-Parametric Safety Stock Modeling (Assignment 2)

## Question 1 — Required Transformation

Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.

### Answer 1:

#### Data Transformation

In order to aggregate the quantity by SKU, we create a new dataframe using the `groupby` and `aggregate` functions across all transactions where `stocking_type == "MTS"`.

Because of the way we are aggregating the quantity and lead time by SKU, it creates multi-level headers. We overwrite this by manually renaming the column headers.


In [49]:
transformed_data = (transaction_data_clean.query('stocking_type == "MTS"')
                    .groupby('sku_number')
                    .aggregate({'order_quantity':['min', 'max', 'mean', 'median', 'var', 'std'], 'lead_time':'mean'})
                    .reset_index())

transformed_data.columns = ['sku_number', 'min_order_quantity', 'max_order_quantity', 'avg_order_quantity', 'median_order_quantity', 'var_order_quantity', 'std_order_quantity', 'avg_lead_time']
transformed_data.sample(5)


,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time
108,3DAC7A33,1,25,8.544118,6.5,36.729368,6.060476,14.000
54,202A9197,1,83,19.254000,15.0,240.911307,15.521318,18.648
13,0707489E,1,34,9.387283,8.0,51.227047,7.157307,14.000
420,D6C550F0,1,29,8.629213,7.0,47.417773,6.886056,14.000
461,E7561C81,1,33,9.274775,8.0,45.159451,6.720078,14.000


## Question 2 — Safety Stock Calculation (Parametric Models)

Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. You will need to find the z-score of the given service level coefficients, for which you can use the ppf function of SciPy.Stat module of the SciPy package, or simply use the numeric values from any standard normal distribution function.

### Answer 2:

#### Type of Distribution

##### Using Coefficient of Variation Rule for Normal Distribution

First, we want to check to see if our inventory is normally distributed or if a different distribution should be used. We will add another column for the coefficient of variation. The formula for the coefficient of variation is as follows:

$$\text{CV} = \frac{\text{Std Dev}}{\text{Mean}}$$

If the CV is less than 30%, demand is relatively stable and predictable, relative to its volume, indicating that we can most likely use a normal distribution.

Now, we calculate the CV and find the SKUs which we can apply a normal distribution.


In [50]:
transformed_data['cv'] = transformed_data['std_order_quantity']/transformed_data['avg_order_quantity']
transformed_data.query('cv <= 0.3')


,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv
1,00874431,1,164,98.524059,100.0,558.627557,23.635303,27.710120,0.239894
8,048B94FE,19,165,99.866627,100.0,418.892640,20.466867,28.000000,0.204942
18,0839933A,27,169,99.800364,100.0,408.627395,20.214534,28.000000,0.202550
25,0D4A17C7,1,161,95.801063,99.0,697.599503,26.412109,27.413223,0.275697
28,123D5CE8,30,161,100.403285,101.0,406.071590,20.151218,28.000000,0.200703
...,...,...,...,...,...,...,...,...,...
479,F633F4BB,36,171,100.611180,100.5,370.224114,19.241209,28.000000,0.191243
485,FA9DDBC6,1,164,95.920863,99.0,736.716325,27.142519,27.356945,0.282968
486,FB796951,1,164,95.587519,99.0,737.135357,27.150237,27.357196,0.284035
491,FED6CA5E,37,165,98.974586,100.0,375.621471,19.380956,28.000000,0.195818


##### Variance to Mean Ratio

Next, we will use a variance to mean ratio to determine whether the SKUs have a binomial, poisson, or negative binomial distribution, using different 'generalized' thresholds.

$$\text{VMR} = \frac{\text{Variance}}{\text{Mean}}$$


In [51]:
transformed_data['VMR'] = transformed_data['var_order_quantity']/transformed_data['avg_order_quantity']


##### Filtering for Binomial & Poisson Distributions

After filtering for any SKUs with a high coefficient of variation and variance to mean ratio less than 0.8 (Binomial) or between 0.8 and 1.2 (Poisson), we identified 0 SKUs fitting Binomial or Poisson.


In [52]:
# Binomial check (VMR < 0.8)
transformed_data.query('cv > 0.3 and VMR < 0.8')


,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR


In [53]:
# Poisson check (0.8 <= VMR < 1.2)
transformed_data.query('cv > 0.3 and VMR >= 0.8 and VMR < 1.2')


,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR


##### Filtering for Negative Binomial Distributions

After filtering for any SKUs with a high coefficient of variation and a variance to mean ratio greater than 1.2, we identified the remaining non-normally distributed SKUs as having a negative binomial distribution. This means that most SKUs have a fairly low demand/order quantity but experience erratic or unpredictable high-quantity spikes. Using a safety stock calculation which assumes a normal distribution, we may experience more stockouts on these SKUs.


In [54]:
transformed_data.query('cv > 0.3 and VMR >= 1.2')


,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR
0,00478A8F,1,166,80.293206,92.0,1575.097842,39.687502,25.714546,0.494282,19.616826
2,018AA89B,1,21,8.262500,7.0,32.120095,5.667459,14.000000,0.685925,3.887455
3,01FAD69F,1,23,8.683544,6.0,41.347290,6.430186,14.000000,0.740502,4.761568
4,02452E5E,1,87,19.990000,17.0,265.543759,16.295513,19.425000,0.815183,13.283830
5,02CFF6A9,1,88,22.865672,20.0,252.631602,15.894389,21.000000,0.695120,11.048510
...,...,...,...,...,...,...,...,...,...,...
490,FE1EE752,1,43,9.906250,8.0,58.965998,7.678932,14.000000,0.775160,5.952404
492,FEE42AC0,1,28,8.864583,7.0,40.644627,6.375314,14.000000,0.719189,4.585058
493,FF42B9D4,1,29,10.714286,10.0,44.154135,6.644858,14.000000,0.620187,4.121053
495,FF9E44A8,1,21,8.672131,8.0,25.624044,5.062020,14.000000,0.583711,2.954757


#### Find z-scores for Percentiles for Normal Distributions


In [55]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)


#### Calculating Safety Stock for Normal Distributions

Now we use the following function to calculate the safety stock for each SKU:

$$SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

We also want to display whole numbers for these, so we will use the floor division (`//`) and remainder (`%`) functions to achieve this. The floor division will give us the 'rounded' integer. We also use the remainder function, along with a logical operator to return a value of 0 or 1. If the remainder > 0, then the result should be 1. If the remainder is 0, then the result of that subcalculation would be 0. We also want to change the data type to `int64`.


In [56]:
# Safety Stock at 75%
raw_ss_75_norm = z_75 * transformed_data['std_order_quantity'] * np.sqrt(transformed_data['avg_lead_time'])
transformed_data['ss_75_norm'] = (raw_ss_75_norm // 1 + (raw_ss_75_norm % 1 > 0)).astype('int64')

# Safety Stock at 90%
raw_ss_90_norm = z_90 * transformed_data['std_order_quantity'] * np.sqrt(transformed_data['avg_lead_time'])
transformed_data['ss_90_norm'] = (raw_ss_90_norm // 1 + (raw_ss_90_norm % 1 > 0)).astype('int64')

# Safety Stock at 95%
raw_ss_95_norm = z_95 * transformed_data['std_order_quantity'] * np.sqrt(transformed_data['avg_lead_time'])
transformed_data['ss_95_norm'] = (raw_ss_95_norm // 1 + (raw_ss_95_norm % 1 > 0)).astype('int64')

transformed_data.sample(5)


,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,ss_75_norm,ss_90_norm,ss_95_norm
431,DC272A30,35,165,99.873950,100.0,408.059777,20.200490,28.000000,0.202260,4.085748,73,137,176
106,3CB1531F,1,167,91.975344,97.0,1016.110120,31.876482,26.827982,0.346576,11.047636,112,212,272
349,B014F2EB,1,171,97.930420,99.0,586.763061,24.223193,27.661875,0.247351,5.991632,86,164,210
5,02CFF6A9,1,88,22.865672,20.0,252.631602,15.894389,21.000000,0.695120,11.048510,50,94,120
480,F8729E1B,1,34,9.590909,8.0,47.095089,6.862586,14.000000,0.715530,4.910388,18,33,43


#### Calculating Safety Stock for Negative Binomial Distributions

For the SKUs with a negative binomial distribution, we want to identify the reorder point and average demand during lead time to calculate safety stock. The formula is as follows:

$$\text{Safety Stock} = ROP - \mu_{LT}$$

To obtain the reorder point, we use the `scipy.stats.nbinom.ppf` function.

To use the function, we calculate parameters $n$ and $p$ via the Method of Moments:

$$n = \frac{\mu_{LT} \times p}{1 - p}$$
$$p = \frac{\mu_{LT}}{\sigma^2_{LT}}$$
$$\mu_{LT} = \mu_{\text{daily}} \times \text{Lead Time}$$
$$\sigma^2_{LT} = \sigma^2_{\text{daily}} \times \text{Lead Time}$$


In [57]:
transformed_data['mean_during_lt'] = transformed_data['avg_order_quantity']*transformed_data['avg_lead_time']
transformed_data['var_during_lt'] = transformed_data['var_order_quantity']*transformed_data['avg_lead_time']

transformed_data['p_parameter'] = transformed_data['mean_during_lt'] / transformed_data['var_during_lt']
transformed_data['n_parameter'] = ((transformed_data['mean_during_lt'] * transformed_data['p_parameter']) / (1 - transformed_data['p_parameter']))

transformed_data['rop_75'] = stats.nbinom.ppf(0.75, transformed_data['n_parameter'], transformed_data['p_parameter'])
transformed_data['rop_90'] = stats.nbinom.ppf(0.90, transformed_data['n_parameter'], transformed_data['p_parameter'])
transformed_data['rop_95'] = stats.nbinom.ppf(0.95, transformed_data['n_parameter'], transformed_data['p_parameter'])

raw_ss_75_bn = transformed_data['rop_75'] - transformed_data['mean_during_lt']
raw_ss_90_bn = transformed_data['rop_90'] - transformed_data['mean_during_lt']
raw_ss_95_bn = transformed_data['rop_95'] - transformed_data['mean_during_lt']

transformed_data['ss_75_bn'] = (raw_ss_75_bn // 1 + (raw_ss_75_bn % 1 > 0)).astype('int64')
transformed_data['ss_90_bn'] = (raw_ss_90_bn // 1 + (raw_ss_90_bn % 1 > 0)).astype('int64')
transformed_data['ss_95_bn'] = (raw_ss_95_bn // 1 + (raw_ss_95_bn % 1 > 0)).astype('int64')


#### Finalized Safety Stock Column (np.where)

Since we want to consolidate the 'correct' safety stock for each SKU, we use `np.where()` to assign the Normal safety stock if CV <= 0.3, and Negative Binomial safety stock if CV > 0.3.


In [58]:
transformed_data['ss_75'] = np.where(transformed_data['cv'] <= 0.3, transformed_data['ss_75_norm'], transformed_data['ss_75_bn'])
transformed_data['ss_90'] = np.where(transformed_data['cv'] <= 0.3, transformed_data['ss_90_norm'], transformed_data['ss_90_bn'])
transformed_data['ss_95'] = np.where(transformed_data['cv'] <= 0.3, transformed_data['ss_95_norm'], transformed_data['ss_95_bn'])

transformed_data.sample(5)


,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,...,n_parameter,rop_75,rop_90,rop_95,ss_75_bn,ss_90_bn,ss_95_bn,ss_75,ss_90,ss_95
127,44431D53,1,27,8.674699,8.0,38.905084,6.237394,14.000000,0.719033,4.484892,...,34.849228,136.0,152.0,162.0,15,31,41,15,31,41
316,9F3EF113,1,32,8.493671,7.0,42.176241,6.494324,14.000000,0.764607,4.965608,...,29.985664,134.0,151.0,161.0,16,33,43,16,33,43
440,DEDE31A1,1,36,11.005682,10.5,49.925894,7.065826,21.198864,0.642016,4.536374,...,65.973769,254.0,276.0,289.0,21,43,56,21,43,56
138,49C97559,1,29,10.150685,9.0,36.240868,6.020039,14.000000,0.593067,3.570288,...,55.289366,157.0,172.0,181.0,15,30,39,15,30,39
351,B11004B8,1,34,11.910112,11.0,65.393005,8.086594,28.000000,0.678969,5.490545,...,74.263406,361.0,389.0,407.0,28,56,74,28,56,74


## Question 3 — Parametric Safety Stock Distribution

Now that we have the safety stock values, write appropriate code to answer the following questions. For this question, focus on the 95% service level coefficient.

- Which SKU has the largest safety stock?
- Which SKU has the smallest safety stock?
- What is the average safety stock value across all SKUs?

### Answer 3:
Before answering questions, I want to create a copy of my transformed data into a neat and clean table with just the SKU and safety stocks for each method (since this would also be a way to validate the data).


In [59]:
selected_columns = ['sku_number', 'ss_95_norm', 'ss_95_bn', 'ss_95']
data_95_ss = transformed_data[selected_columns].copy()


#### Largest Safety Stock

In order to find the SKU with the largest safety stock, using a 95% service level agreement (SLA), we sort the values by DESCENDING values from the 'ss_95' column.

The SKU with the largest safety stock is AC3ADABF, since it uses the negative binomial distribution. However, if assuming all SKUs had a normal distribution, the SKU with the highest safety stock is 089FA02B.


In [60]:
data_95_ss.sort_values('ss_95', ascending = False).head(5)


,sku_number,ss_95_norm,ss_95_bn,ss_95
342,AC3ADABF,352,370,370
20,089FA02B,353,366,366
471,F0610752,351,362,362
489,FDDDA064,349,360,360
156,56D83D7A,348,359,359


In [61]:
data_95_ss.sort_values('ss_95_norm', ascending=False).head(5)


,sku_number,ss_95_norm,ss_95_bn,ss_95
20,089FA02B,353,366,366
342,AC3ADABF,352,370,370
471,F0610752,351,362,362
489,FDDDA064,349,360,360
156,56D83D7A,348,359,359


#### Smallest Safety Stock

In order to find the SKU with the smallest safety stock, using a 95% service level agreement (SLA), we sort the values by ASCENDING values from the 'ss_95' column.

The SKU with the smallest safety stock is 58F77D5E, using both methods.


In [62]:
data_95_ss.sort_values('ss_95', ascending = True).head(5)


,sku_number,ss_95_norm,ss_95_bn,ss_95
163,58F77D5E,29,30,30
61,243ECBFB,31,33,33
495,FF9E44A8,32,33,33
49,1F900448,33,34,34
384,C6D49812,34,35,35


In [63]:
data_95_ss.sort_values('ss_95_norm', ascending = True).head(5)


,sku_number,ss_95_norm,ss_95_bn,ss_95
163,58F77D5E,29,30,30
61,243ECBFB,31,33,33
495,FF9E44A8,32,33,33
49,1F900448,33,34,34
384,C6D49812,34,35,35


#### Average Safety Stock

To obtain the average safety stock, we can just use the `.describe()` function to find the mean.

The average safety stock is about ~123 units, when doing a mix between normal and negative binomial distributions, ~124 units when looking only at a negative binomial distribution, and ~120 units when looking only at a normal distribution.


In [64]:
data_95_ss.describe()


,ss_95_norm,ss_95_bn,ss_95
count,497.000000,497.000000,497.000000
mean,119.997988,123.674044,123.146881
std,96.063036,97.887579,97.463251
min,29.000000,30.000000,30.000000
25%,41.000000,44.000000,44.000000
50%,66.000000,69.000000,69.000000
75%,176.000000,178.000000,176.000000
max,353.000000,370.000000,370.000000


## SIDEQUEST — Non-Parametric (Empirical Quantile) Safety Stock Calculation

While we already performed the above analysis, we will implement the **SUGGESTED** sidequest of calculating non-parametric safety stock using the empirical quantile formula (because one of our team members went DEEP down the rabbit hole and didn't see the side quest until after she already completed the above exercise). We know the following safety stock formula for parametric (normal) distribution:

$$SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

For a non-parametric safety stock calculation, we look at our dataset, finding the stock quantity at the 75th, 90th and 95th percentile. Rather than using a z score with the standard deviation, we are essentially calculating the nth percentile of our dataset and subtracting from the mean to find the **spread**. In other words, the z-score and standard deviation is your *demand buffer*. We are just replacing the demand buffer with a different variation, where we are getting the nth quartile and subtracting from the mean.

$$\text{Safety Stock} = (\text{Quantile}_{CSL}(d) - \bar{d}) \times \sqrt{LT}$$

Since the quantiles need aggregated data by SKU, we create 3 new aggregated series with the quantile calculation at each service level. We also discovered that we had to use the float function, since we needed to transform the array to Python scalars, due to a type error. Then, we merge this information with the existing 'transformed_data' table (pd.merge, since we already have the sku_number in both tables), then calculate the safety stock using the above formula with the appropriate rounding.


In [65]:
q75 = transaction_data_clean.groupby('sku_number')['order_quantity'].aggregate(lambda x: float(mquantiles(x, prob=0.75)[0])).reset_index(name = 'q75_order_quantity')
q90 = transaction_data_clean.groupby('sku_number')['order_quantity'].aggregate(lambda x: float(mquantiles(x, prob=0.90)[0])).reset_index(name = 'q90_order_quantity')
q95 = transaction_data_clean.groupby('sku_number')['order_quantity'].aggregate(lambda x: float(mquantiles(x, prob=0.95)[0])).reset_index(name = 'q95_order_quantity')

transformed_data = pd.merge(transformed_data, q75, on = 'sku_number', how = 'left')
transformed_data = pd.merge(transformed_data, q90, on = 'sku_number', how = 'left')
transformed_data = pd.merge(transformed_data, q95, on = 'sku_number', how = 'left')

raw_ss_np_75 = (transformed_data['q75_order_quantity'] - transformed_data['avg_order_quantity']) * np.sqrt(transformed_data['avg_lead_time'])
raw_ss_np_90 = (transformed_data['q90_order_quantity'] - transformed_data['avg_order_quantity']) * np.sqrt(transformed_data['avg_lead_time'])
raw_ss_np_95 = (transformed_data['q95_order_quantity'] - transformed_data['avg_order_quantity']) * np.sqrt(transformed_data['avg_lead_time'])

transformed_data['ss_np_75'] = (raw_ss_np_75 // 1 + (raw_ss_np_75 % 1 > 0)).astype('int64')
transformed_data['ss_np_90'] = (raw_ss_np_90 // 1 + (raw_ss_np_90 % 1 > 0)).clip(lower=0).astype('int64')
transformed_data['ss_np_95'] = (raw_ss_np_95 // 1 + (raw_ss_np_95 % 1 > 0)).clip(lower=0).astype('int64')

transformed_data.sample(5)


,sku_number,min_order_quantity,max_order_quantity,avg_order_quantity,median_order_quantity,var_order_quantity,std_order_quantity,avg_lead_time,cv,VMR,...,ss_95_bn,ss_75,ss_90,ss_95,q75_order_quantity,q90_order_quantity,q95_order_quantity,ss_np_75,ss_np_90,ss_np_95
65,25D9FC60,1,27,9.320755,7.0,44.042035,6.636417,14.000000,0.712004,4.725158,...,44,16,33,44,14.0,19.00,22.00,18,37,48
24,0CD5B7B2,1,25,8.591195,7.0,33.774859,5.811614,14.000000,0.676462,3.931334,...,38,14,29,38,111.0,123.00,131.00,384,429,459
362,B825F588,1,26,9.571429,7.5,59.549053,7.716803,14.000000,0.806233,6.221543,...,51,18,38,51,15.0,23.00,24.39,21,51,56
110,3E7194CB,1,36,12.058394,10.0,62.908330,7.931477,28.000000,0.657756,5.216974,...,72,28,55,72,16.0,23.00,27.48,21,58,82
314,9F0369DD,1,162,80.329892,93.0,1558.439007,39.477069,25.703895,0.491437,19.400487,...,340,132,261,340,107.0,121.28,130.00,136,208,252


### Re-answering Question 3 for non-parametric (quantile) calculation

We will re-select new columns and perform sorting by the largest, smallest, and average safety stock.

When looking at the safety stock using the non-parametric (quantile) safety stock calculation, we see that the SKU is AC3ADABF, which matched our answer for the negative binomial distribution. However, the safety stock is still much lower than our other calculations, and only 3 of the 5 top SKUs appeared from our previous list.

When looking at the smallest safety stock, SKU 58F77D5E is still the smallest across all three methods. However, the value for the non-parametric method resulted in a higher value in comparison to the other 2 methods. Additionally, we again see that only 3 of the 5 bottom SKUs appeared in our non-parametric list, compared to our previous list.

Lastly, the average safety stock is reduced down to ~114 units, which is a much more conservative value in comparison to both methods. If the organization wants to be more conservative with their approach, it may be best to use the non-parametric method for safety stock calculation, the average safety stock is much lower than any of the other methods that we have attempted.


In [66]:
selected_columns = ['sku_number', 'ss_95_norm', 'ss_95_bn', 'ss_95', 'ss_np_95']
data_95_ss = transformed_data[selected_columns].copy()

# Top 5 by Non-Parametric Safety Stock
data_95_ss.sort_values('ss_np_95', ascending = False).head(5)


,sku_number,ss_95_norm,ss_95_bn,ss_95,ss_np_95
58,22CCC5EC,67,69,69,632
14,072EBE1C,67,70,70,623
68,27ECF714,64,67,67,582
364,B9E45FC2,57,59,59,550
307,99E3E090,54,57,57,534


In [67]:
# Bottom 5 by Non-Parametric Safety Stock
data_95_ss.sort_values('ss_np_95', ascending = True).head(5)


,sku_number,ss_95_norm,ss_95_bn,ss_95,ss_np_95
49,1F900448,33,34,34,35
384,C6D49812,34,35,35,35
61,243ECBFB,31,33,33,35
380,C565BADC,40,42,42,36
338,AADC7D9C,38,40,40,36


In [68]:
# Summary statistics across all 95% service level methods
data_95_ss.describe()


,ss_95_norm,ss_95_bn,ss_95,ss_np_95
count,497.000000,497.000000,497.000000,497.000000
mean,119.997988,123.674044,123.146881,160.237425
std,96.063036,97.887579,97.463251,130.685722
min,29.000000,30.000000,30.000000,35.000000
25%,41.000000,44.000000,44.000000,53.000000
50%,66.000000,69.000000,69.000000,138.000000
75%,176.000000,178.000000,176.000000,196.000000
max,353.000000,370.000000,370.000000,632.000000


## References

CASRAI. (n.d.). Negative binomial regression overdispersion. CASRAI Guides.

Hakyimlab. (2026). TypeError: only 0-dimensional arrays can be converted to Python scalars with numpy >= 2.x (Issue No. 225).

Oracle. (n.d.). NetSuite help center: Inventory management guidelines. Oracle Help Center.

Pandas development team. (n.d.). Pandas.DataFrame.aggregate. pandas.

Stack Overflow. (2022). How to substring values from a pandas df.

Zaiontz, C. (n.d.). Method of moments: Negative binomial distribution. Real Statistics Using Excel.
